# V5 quick baselines and controls
Run fixed majority and logistic baselines on the same 12 held-out participants and seeds 45/62 as the completed CNN confirmation. This performs no neural-network training. A CPU or GPU Colab runtime works. Loading data and saving the files usually take a few minutes.
The logistic features are a nonzero word mean for each of 126 channels plus 126 nonzero-word fractions. Coverage-only logistic uses just those fractions. Coverage is a proxy that includes word padding and missing fixations, so it can reflect sentence length as well as missingness. Readers are separate training samples; features are not averaged across readers.
The sentence-label shuffle assigns a balanced random label mapping to the same sentence IDs across participants. Its score concerns transferable sentence fingerprints, not sentiment. Two random mappings are a diagnostic rather than a formal permutation test.

## 1. Setup
Code is fixed at tag `v5-fast-baselines-v1`. No package installation is needed in the standard Colab runtime.

In [ ]:
import os, sys, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v5-fast'
run_ref = 'v5-fast-baselines-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'Use the clean fixed revision.'
print('Ready. Git commit:', commit, flush=True)


## 2. Run the fast baselines
Four models × two seeds × 12 participants = 96 result rows. Every completed result is saved. The fit pool is the same ten participants as the CNN confirmation; one additional participant is reserved for validation even though fixed logistic regression does not tune on it. Standardization uses only the fit pool. Logistic results can repeat across seeds; these are paired comparisons, not independent participant samples.

In [ ]:
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v5_fast_baselines_45_62')
command = [sys.executable, '-u', '-B', 'src/v5_baselines.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--out-dir', out_dir, '--config', 'configs/v5_fast_baselines.json']
print('Starting fixed fast baselines; live output follows:', flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print('Stopped. Completed rows remain saved.', flush=True)
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 3. Compare with the completed CNN
Checks data fingerprints and participant splits before joining the results. The primary comparison uses real sentiment targets. The shuffled-label control is shown separately. The majority predictor's one-class output is intentional.

In [ ]:
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
assert len(metrics) == 96 and not metrics[['model', 'seed', 'participant']].duplicated().any()
cnn_dir = os.path.join(base, 'v4_low_lr_confirmation_45_62')
cnn = pd.read_csv(os.path.join(cnn_dir, 'fold_metrics.csv'))
assert len(cnn) == 24 and not cnn[['seed', 'participant']].duplicated().any()
with open(os.path.join(out_dir, 'run_metadata.json')) as stream:
    baseline_meta = json.load(stream)
with open(os.path.join(cnn_dir, 'run_metadata.json')) as stream:
    cnn_meta = json.load(stream)
assert baseline_meta['definition']['data_files'] == cnn_meta['definition']['data_files'], 'Data fingerprints differ.'
real = metrics[metrics.target_type == 'sentiment'].copy()
fields = ['seed', 'participant', 'macro_f1', 'collapsed', 'validation_participant', 'fit_participants', 'n_fit', 'n_validation', 'n_test']
paired = real.merge(cnn[fields], on=['seed', 'participant'], suffixes=('_baseline', '_cnn'), validate='many_to_one')
assert len(paired) == len(real), 'Some participant/seed pairs did not match.'
for name in ['validation_participant', 'fit_participants', 'n_fit', 'n_validation', 'n_test']:
    assert (paired[name + '_baseline'] == paired[name + '_cnn']).all(), 'Split mismatch: ' + name
paired['macro_f1_minus_cnn'] = paired.macro_f1_baseline - paired.macro_f1_cnn
paired.to_csv(os.path.join(out_dir, 'paired_real_target_differences.csv'), index=False)
comparison = real.groupby('model', as_index=False).agg(folds=('macro_f1', 'size'), mean_macro_f1=('macro_f1', 'mean'), collapse_count=('collapsed', 'sum'))
cnn_row = pd.DataFrame([{'model': 'original_cnn_lr_0.0001', 'folds': len(cnn), 'mean_macro_f1': cnn.macro_f1.mean(), 'collapse_count': cnn.collapsed.sum()}])
comparison = pd.concat([comparison, cnn_row], ignore_index=True)
comparison.to_csv(os.path.join(out_dir, 'comparison_with_confirmed_cnn.csv'), index=False)
print('Real sentiment targets:')
display(comparison)
print('Shuffled sentence-label control (a different target):')
control = pd.read_csv(os.path.join(out_dir, 'summary_by_model_seed.csv'))
display(control[control.target_type == 'shuffled_sentence_labels'])
